# CoI Estimator Validation: Accuracy, Consistency, and Subsampling Stability

Validates the Co-Information (CoI) measurement tool by comparing three MI estimators on synthetic planted-synergy data:
- **NPEET micd**: Non-parametric entropy estimation toolbox
- **sklearn + custom KSG**: sklearn individual MI + custom Ross 2014 joint MI
- **frbourassa cKDTree**: KSG estimator using scipy cKDTree

The experiment tests whether estimators correctly identify XOR synergy pairs (CoI < 0), consistency between estimators, subsampling stability, and reproducibility across random seeds.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('npeet')
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

In [ ]:
import gc
import json
import math
import os
import sys
import time
from itertools import combinations
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from loguru import logger
from scipy import stats
from scipy.special import digamma
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.neighbors import KDTree, NearestNeighbors

# Configure logging for notebook
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

MASTER_SEED = 42

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-7dffcf-balance-guided-oblique-trees-signed-spec/main/experiment_iter2_coi_estimator_v/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded reference data with {len(data['datasets'])} datasets")
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")
print(f"\nRecommendation: {data['metadata']['recommendation']['best_estimator']}")

## Configuration

All tunable parameters are defined here. Set to minimum values for a quick demo run; original values shown in comments.

In [ ]:
# ============ CONFIGURATION ============
# All tunable parameters — set to MINIMUM values for quick demo.
# Original values shown in comments.

N_EASY = 500          # Original: 10000
N_MEDIUM = 500        # Original: 20000
K_NEIGHBORS = 5       # KSG neighbor parameter (original: 5)
SMOKE_SIZE = 100      # Original: 500
SUBSAMPLE_SIZES = [100, 200, 400]  # Original: [1000, 2000, 5000, 10000, 15000, 20000]
N_REPRO_SEEDS = 2     # Original: 10
REPRO_N_SUB = 200     # Original: 10000
SPONGE_K = 4          # Number of clusters for SPONGE (original: 4)

## Data Generation Primitives

Generates synthetic datasets with known ground-truth synergy/redundancy structure:
- **XOR interaction**: Creates synergistic feature pairs (CoI should be negative)
- **AND interaction**: Creates another type of interaction
- **Redundant copies**: Noisy copies of features (CoI should be positive)

In [ ]:
def xor_interaction(x1: np.ndarray, x2: np.ndarray) -> np.ndarray:
    return np.sign(x1 * x2)

def and_interaction(x1: np.ndarray, x2: np.ndarray) -> np.ndarray:
    return ((x1 > 0) & (x2 > 0)).astype(float)

def make_redundant(x: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    return x + rng.normal(0, sigma, size=x.shape)

def generate_target(contributions, weights, sigma_noise, rng):
    n = contributions[0].shape[0]
    logit = np.zeros(n)
    for c, w in zip(contributions, weights):
        logit += w * (c - c.mean())
    logit += rng.normal(0, sigma_noise, size=n)
    return (logit > 0).astype(int)

def assign_folds(y, n_splits=5, random_state=42):
    folds = np.zeros(len(y), dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for fold_idx, (_, test_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
        folds[test_idx] = fold_idx
    return folds

def gen_easy_2mod_xor(rng: np.random.Generator, n: int = None) -> dict:
    if n is None:
        n = N_EASY
    d = 10
    X = rng.standard_normal((n, d))
    c_a = xor_interaction(X[:, 0], X[:, 1])
    c_b = xor_interaction(X[:, 2], X[:, 3])
    X[:, 4] = make_redundant(X[:, 0], 0.3, rng)
    X[:, 5] = make_redundant(X[:, 2], 0.3, rng)
    y = generate_target([c_a, c_b], [1.0, 1.0], 0.1, rng)
    folds = assign_folds(y)
    meta = {
        "n_samples": n, "n_features": d, "n_modules": 2,
        "ground_truth_modules": [[0, 1], [2, 3]],
        "module_types": ["xor", "xor"],
        "module_weights": [1.0, 1.0],
        "sigma_noise": 0.1,
        "redundant_pairs": [[0, 4], [2, 5]],
        "redundant_sigma": 0.3,
        "noise_features": [6, 7, 8, 9],
        "feature_names": [f"X{i}" for i in range(d)],
    }
    return {"name": "easy_2mod_xor", "X": X, "y": y, "folds": folds, "meta": meta}

def gen_medium_4mod_mixed(rng: np.random.Generator, n: int = None) -> dict:
    if n is None:
        n = N_MEDIUM
    d = 18
    X = rng.standard_normal((n, d))
    c_a = xor_interaction(X[:, 0], X[:, 1])
    c_b = xor_interaction(X[:, 2], X[:, 3])
    c_c = and_interaction(X[:, 4], X[:, 5])
    c_d = and_interaction(X[:, 6], X[:, 7])
    X[:, 8] = make_redundant(X[:, 0], 0.3, rng)
    X[:, 9] = make_redundant(X[:, 2], 0.3, rng)
    X[:, 10] = make_redundant(X[:, 4], 0.3, rng)
    X[:, 11] = make_redundant(X[:, 6], 0.3, rng)
    y = generate_target([c_a, c_b, c_c, c_d], [1.0, 1.0, 2.5, 2.5], 0.2, rng)
    folds = assign_folds(y)
    meta = {
        "n_samples": n, "n_features": d, "n_modules": 4,
        "ground_truth_modules": [[0, 1], [2, 3], [4, 5], [6, 7]],
        "module_types": ["xor", "xor", "and", "and"],
        "module_weights": [1.0, 1.0, 2.5, 2.5],
        "sigma_noise": 0.2,
        "redundant_pairs": [[0, 8], [2, 9], [4, 10], [6, 11]],
        "redundant_sigma": 0.3,
        "noise_features": list(range(12, 18)),
        "feature_names": [f"X{i}" for i in range(d)],
    }
    return {"name": "medium_4mod_mixed", "X": X, "y": y, "folds": folds, "meta": meta}

## CoI Estimators

Three implementations of Co-Information estimation:
1. **NPEET micd** — uses the NPEET library's mixed continuous-discrete MI estimator
2. **sklearn + custom KSG** — sklearn for individual MI, custom Ross 2014 for joint MI
3. **frbourassa cKDTree** — KSG estimator using scipy's cKDTree for efficient neighbor search

CoI(Xi, Xj; Y) = MI(Xi; Y) + MI(Xj; Y) - MI(Xi,Xj; Y)
- CoI < 0 implies synergy (pair is more informative jointly)
- CoI > 0 implies redundancy (features share information)

In [ ]:
# --- ESTIMATOR A: NPEET micd ---

def _npeet_mi_joint(args):
    """Compute joint MI for a pair using NPEET micd."""
    import npeet.entropy_estimators as ee
    X_pair, y_2d_list, k = args
    return ee.micd(X_pair, y_2d_list, k=k)

def compute_coi_npeet(X: np.ndarray, y: np.ndarray, k: int = 5) -> tuple:
    """Compute CoI matrix using NPEET micd for all MI terms (nats)."""
    import npeet.entropy_estimators as ee
    d = X.shape[1]
    logger.debug(f"NPEET: computing CoI for {d} features, n={X.shape[0]}, k={k}")
    y_2d_list = y.reshape(-1, 1).tolist()

    mi_indiv = np.zeros(d)
    for i in range(d):
        mi_indiv[i] = ee.micd(X[:, i].reshape(-1, 1), y_2d_list, k=k)

    pairs = [(i, j) for i in range(d) for j in range(i + 1, d)]
    pair_args = [(np.column_stack([X[:, i], X[:, j]]), y_2d_list, k) for i, j in pairs]
    joint_mi = {}
    # Sequential execution (notebook-safe, small data)
    for idx, arg in enumerate(pair_args):
        try:
            joint_mi[pairs[idx]] = _npeet_mi_joint(arg)
        except Exception:
            joint_mi[pairs[idx]] = 0.0

    coi = np.zeros((d, d))
    for (i, j), jmi in joint_mi.items():
        coi[i, j] = mi_indiv[i] + mi_indiv[j] - jmi
        coi[j, i] = coi[i, j]
    return coi, mi_indiv


# --- ESTIMATOR B: sklearn individual MI + custom Ross 2014 joint MI ---

def joint_mi_cd(X_2d: np.ndarray, y: np.ndarray, k: int = 5) -> float:
    """Custom Ross 2014 MI estimator for 2D continuous X with discrete Y."""
    n = len(y)
    classes = np.unique(y)
    rng_noise = np.random.RandomState(42)
    X_noisy = X_2d + 1e-10 * rng_noise.randn(*X_2d.shape)

    nn_distances = np.full(n, np.inf)
    m = np.zeros(n)
    for c in classes:
        mask = (y == c)
        X_c = X_noisy[mask]
        m[mask] = mask.sum()
        if mask.sum() <= k:
            continue
        nn = NearestNeighbors(n_neighbors=k + 1, metric='chebyshev')
        nn.fit(X_c)
        dists, _ = nn.kneighbors(X_c)
        nn_distances[mask] = dists[:, -1]

    tree = KDTree(X_noisy, metric='chebyshev')
    n_all = np.zeros(n)
    for i in range(n):
        if np.isinf(nn_distances[i]):
            n_all[i] = 1
        else:
            n_all[i] = max(tree.query_radius(X_noisy[i:i+1], r=nn_distances[i],
                                              count_only=True)[0] - 1, 1)

    mi = digamma(k) - np.mean(digamma(m)) + digamma(n) - np.mean(digamma(n_all))
    return max(float(mi), 0.0)

def _sklearn_custom_joint(args):
    X_pair, y, k = args
    return joint_mi_cd(X_pair, y, k)

def compute_coi_sklearn_custom(X: np.ndarray, y: np.ndarray, k: int = 5) -> tuple:
    """Compute CoI using sklearn individual MI + custom Ross 2014 joint MI (nats)."""
    d = X.shape[1]
    logger.debug(f"sklearn+custom: computing CoI for {d} features, n={X.shape[0]}")
    mi_indiv = mutual_info_classif(X, y, n_neighbors=k, random_state=42)

    pairs = [(i, j) for i in range(d) for j in range(i + 1, d)]
    pair_args = [(np.column_stack([X[:, i], X[:, j]]), y, k) for i, j in pairs]
    joint_mi = {}
    for idx, arg in enumerate(pair_args):
        try:
            joint_mi[pairs[idx]] = _sklearn_custom_joint(arg)
        except Exception:
            joint_mi[pairs[idx]] = 0.0

    coi = np.zeros((d, d))
    for (i, j), jmi in joint_mi.items():
        coi[i, j] = mi_indiv[i] + mi_indiv[j] - jmi
        coi[j, i] = coi[i, j]
    return coi, mi_indiv


# --- ESTIMATOR C: frbourassa cKDTree ---

def discrete_continuous_info_fast(d_arr: np.ndarray, c_arr: np.ndarray, k: int = 5) -> float:
    """MI between discrete d_arr and continuous c_arr using cKDTree (nats)."""
    n = len(d_arr)
    classes = np.unique(d_arr)
    rng_noise = np.random.RandomState(42)
    if c_arr.ndim == 1:
        c_arr = c_arr.reshape(-1, 1)
    c_noisy = c_arr + 1e-10 * rng_noise.randn(*c_arr.shape)

    nn_distances = np.full(n, np.inf)
    m = np.zeros(n, dtype=float)
    for cls in classes:
        mask = (d_arr == cls)
        c_cls = c_noisy[mask]
        m_cls = mask.sum()
        m[mask] = m_cls
        if m_cls <= k:
            continue
        tree_cls = cKDTree(c_cls)
        dists, _ = tree_cls.query(c_cls, k=k + 1, p=np.inf)
        nn_distances[mask] = dists[:, -1]

    tree_all = cKDTree(c_noisy)
    n_all = np.ones(n)
    for i in range(n):
        if not np.isinf(nn_distances[i]):
            count = tree_all.query_ball_point(c_noisy[i], r=nn_distances[i], p=np.inf)
            n_all[i] = max(len(count) - 1, 1)

    mi = digamma(k) - np.mean(digamma(m)) + digamma(n) - np.mean(digamma(n_all))
    return max(float(mi), 0.0)

def _frbourassa_joint(args):
    X_pair, y, k = args
    return discrete_continuous_info_fast(y, X_pair, k=k)

def compute_coi_frbourassa(X: np.ndarray, y: np.ndarray, k: int = 5) -> tuple:
    """Compute CoI using frbourassa cKDTree for all MI terms (nats)."""
    d = X.shape[1]
    logger.debug(f"frbourassa: computing CoI for {d} features, n={X.shape[0]}")
    mi_indiv = np.zeros(d)
    for i in range(d):
        mi_indiv[i] = discrete_continuous_info_fast(y, X[:, i], k=k)

    pairs = [(i, j) for i in range(d) for j in range(i + 1, d)]
    pair_args = [(np.column_stack([X[:, i], X[:, j]]), y, k) for i, j in pairs]
    joint_mi = {}
    for idx, arg in enumerate(pair_args):
        try:
            joint_mi[pairs[idx]] = _frbourassa_joint(arg)
        except Exception:
            joint_mi[pairs[idx]] = 0.0

    coi = np.zeros((d, d))
    for (i, j), jmi in joint_mi.items():
        coi[i, j] = mi_indiv[i] + mi_indiv[j] - jmi
        coi[j, i] = coi[i, j]
    return coi, mi_indiv

## SPONGE Clustering & Comparison Metrics

- **SPONGE**: Signed graph spectral clustering that handles both positive (redundancy) and negative (synergy) edges
- **Comparison metrics**: Kendall tau, sign agreement, Pearson correlation of absolute values
- **Ground truth validation**: Checks if estimated CoI signs match known synergy/redundancy structure

In [ ]:
def run_sponge(coi_matrix: np.ndarray, k: int, tau_p: float = 1.0, tau_n: float = 1.0) -> np.ndarray:
    """Minimal SPONGE_sym implementation using scipy.linalg.eigh."""
    from scipy.linalg import eigh
    d = coi_matrix.shape[0]
    if k >= d:
        k = max(2, d // 2)

    A_pos = np.maximum(coi_matrix, 0)
    A_neg = np.maximum(-coi_matrix, 0)
    D_pos = np.diag(A_pos.sum(axis=1))
    D_neg = np.diag(A_neg.sum(axis=1))
    L_pos = D_pos - A_pos
    L_neg = D_neg - A_neg

    eps = 1e-10
    d_pos_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(np.diag(D_pos), eps)))
    d_neg_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(np.diag(D_neg), eps)))
    L_sym_pos = d_pos_inv_sqrt @ L_pos @ d_pos_inv_sqrt
    L_sym_neg = d_neg_inv_sqrt @ L_neg @ d_neg_inv_sqrt

    A_mat = L_sym_pos + tau_n * np.eye(d)
    B_mat = L_sym_neg + tau_p * np.eye(d)
    B_mat += eps * np.eye(d)

    try:
        eigenvalues, eigenvectors = eigh(A_mat, B_mat, subset_by_index=[0, k - 1])
    except Exception:
        try:
            from scipy.linalg import eig
            eigenvalues_all, eigenvectors_all = eig(A_mat, B_mat)
            idx_sort = np.argsort(eigenvalues_all.real)[:k]
            eigenvectors = eigenvectors_all.real[:, idx_sort]
        except Exception:
            return np.zeros(d, dtype=int)

    norms = np.linalg.norm(eigenvectors, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-10)
    V = eigenvectors / norms
    km = KMeans(n_clusters=k, n_init=20, random_state=42, max_iter=300)
    return km.fit_predict(V)


def compute_comparison_metrics(coi_A, coi_B, name_A, name_B):
    """Compare two CoI matrices using rank correlation, sign agreement, Pearson."""
    d = coi_A.shape[0]
    triu_idx = np.triu_indices(d, k=1)
    vals_A, vals_B = coi_A[triu_idx], coi_B[triu_idx]

    tau, tau_p = stats.kendalltau(vals_A, vals_B)
    eps = 0.001
    signs_A = np.where(np.abs(vals_A) < eps, 0, np.sign(vals_A))
    signs_B = np.where(np.abs(vals_B) < eps, 0, np.sign(vals_B))
    sign_agree = float(np.mean(signs_A == signs_B))

    abs_A, abs_B = np.abs(vals_A), np.abs(vals_B)
    pearson_r = float(np.corrcoef(abs_A, abs_B)[0, 1]) if np.std(abs_A) > 1e-10 and np.std(abs_B) > 1e-10 else 0.0

    return {
        "kendall_tau": round(float(tau), 4),
        "kendall_p": round(float(tau_p), 6),
        "sign_agreement": round(sign_agree, 4),
        "abs_value_pearson": round(pearson_r, 4),
    }


def validate_ground_truth(coi, mi_indiv, meta, estimator_name):
    """Validate CoI signs against ground truth expectations."""
    results = {"estimator": estimator_name, "checks": []}
    threshold, mi_zero_thresh = 0.01, 0.02

    synergy_correct = synergy_total = 0
    for mod, mtype in zip(meta["ground_truth_modules"], meta["module_types"]):
        if "xor" in mtype and len(mod) >= 2:
            i, j = mod[0], mod[1]
            val = coi[i, j]
            correct = val < -threshold
            results["checks"].append({"type": "synergy", "pair": [i, j], "module_type": mtype,
                "coi_value": round(float(val), 6), "correct_sign": bool(correct)})
            synergy_total += 1
            if correct: synergy_correct += 1

    redundant_correct = redundant_total = 0
    for pair in meta.get("redundant_pairs", []):
        i, j = pair
        val = coi[i, j]
        correct = val > threshold
        results["checks"].append({"type": "redundancy", "pair": [i, j],
            "coi_value": round(float(val), 6), "correct_sign": bool(correct)})
        redundant_total += 1
        if correct: redundant_correct += 1

    noise_correct = noise_total = 0
    for nf in meta.get("noise_features", [])[:3]:
        for other in [0, 1]:
            if other != nf:
                val = coi[nf, other] if nf < coi.shape[0] and other < coi.shape[0] else 0
                correct = abs(val) < 0.05
                results["checks"].append({"type": "noise", "pair": [nf, other],
                    "coi_value": round(float(val), 6), "near_zero": bool(correct)})
                noise_total += 1
                if correct: noise_correct += 1

    xor_mi_correct = xor_mi_total = 0
    for mod, mtype in zip(meta["ground_truth_modules"], meta["module_types"]):
        if "xor" in mtype:
            for feat in mod[:2]:
                if feat < len(mi_indiv):
                    val = mi_indiv[feat]
                    correct = val < mi_zero_thresh
                    results["checks"].append({"type": "xor_marginal_mi", "feature": feat,
                        "mi_value": round(float(val), 6), "near_zero": bool(correct)})
                    xor_mi_total += 1
                    if correct: xor_mi_correct += 1

    results["summary"] = {
        "synergy_pairs_correct_sign": round(synergy_correct / max(synergy_total, 1), 4),
        "redundant_pairs_correct_sign": round(redundant_correct / max(redundant_total, 1), 4),
        "noise_pairs_near_zero": round(noise_correct / max(noise_total, 1), 4),
        "xor_marginal_mi_near_zero": round(xor_mi_correct / max(xor_mi_total, 1), 4),
        "synergy_total": synergy_total,
        "redundant_total": redundant_total,
    }
    return results

## Run Experiment

Executes the full validation pipeline:
- **Phase 0**: Generate synthetic data with planted synergy/redundancy structure
- **Phase 1**: Smoke test (NPEET on small subset)
- **Phase 2**: Full estimator comparison on both data variants
- **Phase 3**: Subsampling stability analysis
- **Phase 4**: Reproducibility across random seeds

In [ ]:
overall_t0 = time.time()
logger.info("=" * 60)
logger.info("CoI Estimator Validation Experiment")
logger.info("=" * 60)

# ---- PHASE 0: Generate synthetic data ----
logger.info("PHASE 0: Generating synthetic data...")
base_rng = np.random.default_rng(MASTER_SEED)
variant_seeds = [int(base_rng.integers(0, 2**31)) for _ in range(6)]

t0 = time.time()
easy_data = gen_easy_2mod_xor(np.random.default_rng(variant_seeds[0]))
logger.info(f"  easy_2mod_xor: {easy_data['X'].shape} in {time.time()-t0:.2f}s")

t0 = time.time()
medium_data = gen_medium_4mod_mixed(np.random.default_rng(variant_seeds[1]))
logger.info(f"  medium_4mod_mixed: {medium_data['X'].shape} in {time.time()-t0:.2f}s")

# ---- PHASE 1: Smoke test ----
logger.info(f"\nPHASE 1: Smoke test on n={SMOKE_SIZE} subset of easy_2mod_xor...")
sss_smoke = StratifiedShuffleSplit(n_splits=1, train_size=SMOKE_SIZE, random_state=42)
idx_smoke, _ = next(sss_smoke.split(easy_data["X"], easy_data["y"]))
X_smoke, y_smoke = easy_data["X"][idx_smoke], easy_data["y"][idx_smoke]

t0 = time.time()
coi_smoke, mi_smoke = compute_coi_npeet(X_smoke, y_smoke, k=K_NEIGHBORS)
dt_smoke = time.time() - t0
logger.info(f"  NPEET smoke test: {dt_smoke:.1f}s")
logger.info(f"  CoI shape: {coi_smoke.shape}, symmetric: {np.allclose(coi_smoke, coi_smoke.T)}")
logger.info(f"  CoI(0,1): {coi_smoke[0,1]:.4f} (expect < 0, XOR synergy)")
logger.info(f"  CoI(0,4): {coi_smoke[0,4]:.4f} (expect > 0, redundancy)")
assert coi_smoke.shape == (10, 10), f"Wrong shape: {coi_smoke.shape}"
assert np.allclose(coi_smoke, coi_smoke.T, atol=1e-6), "Not symmetric"
logger.info("  Smoke test PASSED")
del X_smoke, y_smoke, coi_smoke, mi_smoke; gc.collect()

# ---- PHASE 2: Full estimator comparison ----
logger.info("\nPHASE 2: Full estimator comparison...")
estimator_comparison = {}

variants = [
    ("easy_2mod_xor", easy_data),
    ("medium_4mod_mixed", medium_data),
]

for variant_name, vdata in variants:
    logger.info(f"\n--- Variant: {variant_name} ---")
    X, y, meta = vdata["X"], vdata["y"], vdata["meta"]
    d, n = X.shape[1], X.shape[0]

    variant_results = {
        "n_samples": n, "n_features": d,
        "estimators": {}, "pairwise_correlations": {},
        "ground_truth_validation": {},
    }

    estimators = [
        ("npeet", compute_coi_npeet),
        ("sklearn_custom", compute_coi_sklearn_custom),
        ("frbourassa", compute_coi_frbourassa),
    ]

    coi_matrices, mi_vectors = {}, {}
    for est_name, est_fn in estimators:
        logger.info(f"  Running {est_name}...")
        t0 = time.time()
        try:
            coi_mat, mi_vec = est_fn(X, y, k=K_NEIGHBORS)
            dt = time.time() - t0
            logger.info(f"  {est_name}: {dt:.1f}s")
            coi_matrices[est_name] = coi_mat
            mi_vectors[est_name] = mi_vec
            variant_results["estimators"][est_name] = {
                "time_seconds": round(dt, 2),
                "coi_matrix": coi_mat.tolist(),
                "mi_individual": mi_vec.tolist(),
            }
        except Exception as e:
            logger.error(f"  {est_name} FAILED: {e}")
            continue

    # Pairwise comparisons
    est_names = list(coi_matrices.keys())
    for metric_type in ["kendall_tau", "sign_agreement", "abs_value_pearson"]:
        variant_results["pairwise_correlations"][metric_type] = {}

    for i_est in range(len(est_names)):
        for j_est in range(i_est + 1, len(est_names)):
            name_a, name_b = est_names[i_est], est_names[j_est]
            comp = compute_comparison_metrics(coi_matrices[name_a], coi_matrices[name_b], name_a, name_b)
            key = f"{name_a}_vs_{name_b}"
            variant_results["pairwise_correlations"]["kendall_tau"][key] = comp["kendall_tau"]
            variant_results["pairwise_correlations"]["sign_agreement"][key] = comp["sign_agreement"]
            variant_results["pairwise_correlations"]["abs_value_pearson"][key] = comp["abs_value_pearson"]
            logger.info(f"    {key}: tau={comp['kendall_tau']}, sign={comp['sign_agreement']}, pearson={comp['abs_value_pearson']}")

    # Ground truth validation
    for est_name in coi_matrices:
        gt = validate_ground_truth(coi_matrices[est_name], mi_vectors[est_name], meta, est_name)
        variant_results["ground_truth_validation"][est_name] = gt
        s = gt["summary"]
        logger.info(f"  GT validation {est_name}: synergy={s['synergy_pairs_correct_sign']}, "
                     f"redundancy={s['redundant_pairs_correct_sign']}, xor_mi={s['xor_marginal_mi_near_zero']}")

    estimator_comparison[variant_name] = variant_results
    del coi_matrices, mi_vectors; gc.collect()

# ---- PHASE 3: Subsampling stability (NPEET on medium_4mod_mixed) ----
logger.info("\nPHASE 3: Subsampling stability...")
X_medium, y_medium = medium_data["X"], medium_data["y"]
d_medium = X_medium.shape[1]
triu_idx = np.triu_indices(d_medium, k=1)

logger.info(f"  Computing full-data reference CoI (n={N_MEDIUM})...")
t0 = time.time()
coi_full, _ = compute_coi_npeet(X_medium, y_medium, k=K_NEIGHBORS)
logger.info(f"  Full-data CoI: {time.time()-t0:.1f}s")
vals_full = coi_full[triu_idx]
labels_full = run_sponge(coi_full, k=SPONGE_K)

valid_subsample_sizes = [s for s in SUBSAMPLE_SIZES if s < N_MEDIUM] + [N_MEDIUM]
stability_results = []

for n_sub in valid_subsample_sizes:
    logger.info(f"  Subsampling n={n_sub}...")
    t0 = time.time()
    if n_sub < N_MEDIUM:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=n_sub, random_state=42)
        idx_sub, _ = next(sss.split(X_medium, y_medium))
        X_sub, y_sub = X_medium[idx_sub], y_medium[idx_sub]
    else:
        X_sub, y_sub = X_medium, y_medium

    coi_sub, _ = compute_coi_npeet(X_sub, y_sub, k=K_NEIGHBORS)
    vals_sub = coi_sub[triu_idx]
    spearman_r = float(stats.spearmanr(vals_sub, vals_full).statistic)

    significant = np.abs(vals_full) > 0.001
    sign_flip_rate = float(np.mean(np.sign(vals_sub[significant]) != np.sign(vals_full[significant]))) if significant.sum() > 0 else 0.0

    labels_sub = run_sponge(coi_sub, k=SPONGE_K)
    ari = float(adjusted_rand_score(labels_sub, labels_full))
    dt = time.time() - t0
    logger.info(f"    n={n_sub}: spearman={spearman_r:.4f}, sign_flip={sign_flip_rate:.4f}, ari={ari:.4f} ({dt:.1f}s)")

    stability_results.append({"n_sub": n_sub, "spearman_r": round(spearman_r, 4),
        "sign_flip_rate": round(sign_flip_rate, 4), "sponge_ari": round(ari, 4)})
    if n_sub < N_MEDIUM:
        del X_sub, y_sub
    del coi_sub; gc.collect()

# ---- PHASE 4: Reproducibility ----
logger.info(f"\nPHASE 4: Reproducibility across {N_REPRO_SEEDS} seeds...")
coi_matrices_per_seed = []
repro_n = min(REPRO_N_SUB, N_MEDIUM)
for seed in range(N_REPRO_SEEDS):
    logger.info(f"  Seed {seed}...")
    t0 = time.time()
    sss = StratifiedShuffleSplit(n_splits=1, train_size=repro_n, random_state=seed)
    idx_sub, _ = next(sss.split(X_medium, y_medium))
    X_sub, y_sub = X_medium[idx_sub], y_medium[idx_sub]
    coi_sub, _ = compute_coi_npeet(X_sub, y_sub, k=K_NEIGHBORS)
    coi_matrices_per_seed.append(coi_sub)
    logger.info(f"    Seed {seed}: {time.time()-t0:.1f}s")
    del X_sub, y_sub; gc.collect()

all_vals = np.stack([m[triu_idx] for m in coi_matrices_per_seed])
mean_per_pair = all_vals.mean(axis=0)
std_per_pair = all_vals.std(axis=0)
cv_per_pair = np.where(np.abs(mean_per_pair) > 1e-6, std_per_pair / np.abs(mean_per_pair), np.nan)
median_cv = float(np.nanmedian(cv_per_pair))

labels_per_seed = [run_sponge(m, k=SPONGE_K) for m in coi_matrices_per_seed]
ari_pairs = [float(adjusted_rand_score(labels_per_seed[i], labels_per_seed[j]))
    for i, j in combinations(range(N_REPRO_SEEDS), 2)]
mean_ari = float(np.mean(ari_pairs)) if ari_pairs else 0.0

logger.info(f"  Median CV: {median_cv:.4f}")
logger.info(f"  Mean pairwise ARI: {mean_ari:.4f}")

total_dt = time.time() - overall_t0
logger.info(f"\nDone! Total time: {total_dt:.1f}s ({total_dt/60:.1f} min)")

## Results Visualization

Visualizing the key findings:
1. **CoI Heatmap**: Shows synergy (blue/negative) and redundancy (red/positive) patterns
2. **Estimator Timing**: Computational cost comparison across estimators
3. **Ground Truth Validation**: Accuracy of each estimator at identifying known structure
4. **Subsampling Stability**: How CoI estimates stabilize with increasing sample size

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# --- Plot 1: CoI Heatmap from easy_2mod_xor (NPEET) ---
ax = axes[0, 0]
easy_est = estimator_comparison.get("easy_2mod_xor", {}).get("estimators", {})
if "npeet" in easy_est:
    coi_mat = np.array(easy_est["npeet"]["coi_matrix"])
    im = ax.imshow(coi_mat, cmap="RdBu_r", vmin=-0.3, vmax=0.3, aspect="equal")
    ax.set_title("CoI Matrix (NPEET, easy_2mod_xor)", fontsize=11)
    ax.set_xlabel("Feature"); ax.set_ylabel("Feature")
    for i in range(coi_mat.shape[0]):
        for j in range(coi_mat.shape[1]):
            if i != j:
                ax.text(j, i, f"{coi_mat[i,j]:.2f}", ha="center", va="center", fontsize=6)
    plt.colorbar(im, ax=ax, shrink=0.8, label="CoI (nats)")

# --- Plot 2: Estimator timing comparison ---
ax = axes[0, 1]
ref_times = data["metadata"]["estimator_comparison_summary"]
variants_for_plot = list(ref_times.keys())
est_names_ref = list(ref_times[variants_for_plot[0]]["estimator_times"].keys())
x_pos = np.arange(len(est_names_ref))
width = 0.35
for vi, vname in enumerate(variants_for_plot[:2]):
    times = [ref_times[vname]["estimator_times"].get(e, 0) for e in est_names_ref]
    ax.bar(x_pos + vi * width, times, width, label=f"Ref: {vname}", alpha=0.7)
ax.set_xticks(x_pos + width / 2)
ax.set_xticklabels(est_names_ref, rotation=15, fontsize=9)
ax.set_ylabel("Time (seconds)")
ax.set_title("Estimator Runtime (Reference Data)", fontsize=11)
ax.legend(fontsize=8)

# --- Plot 3: Ground truth validation ---
ax = axes[1, 0]
gt_metrics = ["synergy_pairs_correct_sign", "xor_marginal_mi_near_zero", "noise_pairs_near_zero"]
gt_labels = ["Synergy\nCorrect", "XOR MI\n~0", "Noise\n~0"]
easy_gt = data["metadata"]["estimator_comparison_summary"].get("easy_2mod_xor", {}).get("ground_truth_summary", {})
x_pos = np.arange(len(gt_metrics))
width = 0.25
for ei, est in enumerate(["npeet", "sklearn_custom", "frbourassa"]):
    if est in easy_gt:
        vals = [easy_gt[est].get(m, 0) for m in gt_metrics]
        ax.bar(x_pos + ei * width, vals, width, label=est, alpha=0.8)
ax.set_xticks(x_pos + width)
ax.set_xticklabels(gt_labels, fontsize=9)
ax.set_ylabel("Fraction Correct")
ax.set_title("Ground Truth Validation (Reference)", fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=8)

# --- Plot 4: Subsampling stability ---
ax = axes[1, 1]
ref_stab = data["metadata"]["subsampling_stability"]["results_by_subsample"]
ns = [r["n_sub"] for r in ref_stab]
spearman_ref = [r["spearman_r"] for r in ref_stab]
ax.plot(ns, spearman_ref, "o-", label="Reference", color="steelblue", linewidth=2)

if stability_results:
    ns_demo = [r["n_sub"] for r in stability_results]
    spearman_demo = [r["spearman_r"] for r in stability_results]
    ax.plot(ns_demo, spearman_demo, "s--", label="Demo run", color="coral", linewidth=2)

ax.set_xlabel("Subsample Size (n)")
ax.set_ylabel("Spearman r vs Full Data")
ax.set_title("Subsampling Stability (NPEET)", fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("coi_validation_results.png", dpi=120, bbox_inches="tight")
plt.show()

# --- Summary Table ---
print("\n" + "=" * 70)
print("SUMMARY: CoI Estimator Validation Results")
print("=" * 70)
print(f"\nRecommended estimator: {data['metadata']['recommendation']['best_estimator']}")
print(f"Recommended k-neighbors: {data['metadata']['recommendation']['recommended_k_neighbors']}")
print(f"Minimum stable subsample size: {data['metadata']['recommendation']['minimum_subsample_size']}")
print(f"\nPairwise correlations (easy_2mod_xor reference):")
pw = data["metadata"]["estimator_comparison_summary"]["easy_2mod_xor"]["pairwise_correlations"]
print(f"  {'Pair':<35} {'Kendall tau':>11} {'Sign agree':>11} {'Pearson |r|':>11}")
for key in pw["kendall_tau"]:
    print(f"  {key:<35} {pw['kendall_tau'][key]:>11.4f} {pw['sign_agreement'][key]:>11.4f} {pw['abs_value_pearson'][key]:>11.4f}")

print(f"\nReproducibility (n={data['metadata']['reproducibility']['n_sub']}, "
      f"{data['metadata']['reproducibility']['n_seeds']} seeds):")
print(f"  Median CV: {data['metadata']['reproducibility']['median_cv']:.4f}")
print(f"  Mean pairwise ARI: {data['metadata']['reproducibility']['mean_pairwise_ari']:.4f}")
print(f"  Fraction stable sign: {data['metadata']['reproducibility']['frac_stable_sign_pairs']:.4f}")
print("\nKey finding: sklearn_custom and frbourassa produce identical results (tau=1.0)")
print("NPEET is fastest and recommended despite moderate rank agreement with KSG-based methods.")